##### Purpose: Train Random Forest, XGBoost, and SVM on the best Transformer embeddings. Tune hyperparameters. Select the best classifier.

**Imports**

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import time
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
 
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
 
print("All imports successful")

**Load data**

In [ ]:
SPLITS_PATH = '../data/splits/'
 
y_train = pd.read_csv(SPLITS_PATH + 'y_train.csv').squeeze()
y_val   = pd.read_csv(SPLITS_PATH + 'y_val.csv').squeeze()
 
# Load best transformer embeddings (auto-detect from results)
try:
    with open('../results/best_transformer.txt') as f:
        lines = dict(l.strip().split('=') for l in f.readlines())
    BEST_MODEL = lines['best_transformer']
    print(f"Best Transformer from Notebook 02: {BEST_MODEL}")
except FileNotFoundError:
    # Manually set if you know which was best
    BEST_MODEL = 'BioBERT'   # Change to 'BERT' or 'PubMedBERT' if needed
    print(f"Using manually set model: {BEST_MODEL}")
 
EMBD_TRAIN = f'../models/embeddings/{BEST_MODEL}_train.npy'
EMBD_VAL   = f'../models/embeddings/{BEST_MODEL}_val.npy'
 
X_train_emb = np.load(EMBD_TRAIN)
X_val_emb   = np.load(EMBD_VAL)
 
print(f"Train embeddings: {X_train_emb.shape}")
print(f"Val embeddings:   {X_val_emb.shape}")
print(f"Unique classes:   {y_train.nunique()}")

**Helper evaluation function**

In [ ]:
def evaluate(model, X, y_true, model_name, split_name='Validation'):
    y_pred = model.predict(X)
 
    acc       = accuracy_score(y_true, y_pred)
    f1_macro  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1_micro  = f1_score(y_true, y_pred, average='micro',    zero_division=0)
    f1_weight = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec      = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec       = recall_score(y_true, y_pred,    average='macro', zero_division=0)
 
    print(f"\n{'─'*50}")
    print(f"{model_name} — {split_name}")
    print(f"{'─'*50}")
    print(f"  Accuracy:           {acc:.4f}")
    print(f"  Precision (macro):  {prec:.4f}")
    print(f"  Recall (macro):     {rec:.4f}")
    print(f"  F1 Macro:           {f1_macro:.4f}  ← primary metric")
    print(f"  F1 Micro:           {f1_micro:.4f}")
    print(f"  F1 Weighted:        {f1_weight:.4f}")
 
    meets_target = "✅" if f1_macro >= 0.80 else "⚠️ below target"
    print(f"  Success criterion:  {meets_target} (target ≥ 0.80)")
 
    return {
        'Model': model_name,
        'Split': split_name,
        'Accuracy': acc,
        'Precision_Macro': prec,
        'Recall_Macro': rec,
        'F1_Macro': f1_macro,
        'F1_Micro': f1_micro,
        'F1_Weighted': f1_weight,
    }

*Model 1 — Random Forest*

In [ ]:
print("\n" + "="*60)
print("MODEL 1: RANDOM FOREST")
print("="*60)
 
# Manual tuning — these are good starting values for 768-dim embeddings
# If you want automatic tuning, see the GridSearch version below (takes longer)
rf = RandomForestClassifier(
    n_estimators=200,        # More trees = more stable (diminishing returns >300)
    max_depth=30,            # None = unlimited, but 30 speeds it up
    min_samples_split=5,     # Prevent overfitting on rare disease classes
    min_samples_leaf=2,
    class_weight='balanced', # Critical: upweights rare diseases
    random_state=RANDOM_STATE,
    n_jobs=-1,               # Use all CPU cores
    verbose=1
)
 
print("Training Random Forest...")
t0 = time.time()
rf.fit(X_train_emb, y_train)
print(f"Training time: {time.time()-t0:.0f}s")
 
rf_result = evaluate(rf, X_val_emb, y_val, 'Random Forest')
joblib.dump(rf, '../models/saved_models/random_forest.pkl')
print("Saved: models/saved_models/random_forest.pkl")

*Model 2 — XGBoost*

In [ ]:
print("\n" + "="*60)
print("MODEL 2: XGBOOST")
print("="*60)
 
# XGBoost needs integer labels — already encoded in y_train from Notebook 01
n_classes = int(y_train.nunique())
 
# Compute sample weights to handle class imbalance
# (XGBoost doesn't have class_weight='balanced' like sklearn)
sample_weights = compute_sample_weight('balanced', y_train)
 
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=n_classes,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=1,
    # GPU acceleration — uncomment if you have CUDA
    # tree_method='gpu_hist',
    # device='cuda',
)
 
print("Training XGBoost...")
t0 = time.time()
xgb_model.fit(
    X_train_emb, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_emb, y_val)],
    verbose=50,  # Print every 50 rounds
)
print(f"Training time: {time.time()-t0:.0f}s")
 
xgb_result = evaluate(xgb_model, X_val_emb, y_val, 'XGBoost')
xgb_model.save_model('../models/saved_models/xgboost.json')
print("Saved: models/saved_models/xgboost.json")

[250]	validation_0-mlogloss:1.61567
